# Phân tích Lợi nhuận & Tối ưu hóa Chiến lược Chiết khấu — Phase 3

## Executive Summary

Notebook này thực hiện Giai đoạn 3 của dự án DataCo Supply Chain Analytics:

1. **Profit Margin Analysis**: Đánh giá biên lợi nhuận theo Customer Segment và Market để xác định phân khúc sinh lời nhất.
2. **Discount Impact Analysis**: Phân tích hồi quy (Regression) để đo lường tác động của tỷ lệ chiết khấu lên lợi nhuận.
3. **Discount Optimization (LP)**: Dùng Linear Programming (scipy.optimize) tìm mức chiết khấu tối ưu
   cho từng Segment và Market để tối đa hóa lợi nhuận ròng toàn danh mục.

---

**Dữ liệu**: DataCoSupplyChainDataset.csv — mount từ Google Drive.

**Framework**: Descriptive Analytics → Regression → Linear Programming Optimization.


## Bước 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')


## Bước 1: Import Thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.optimize import linprog
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12
print('✅ Thư viện đã import xong.')


## Bước 2: Load & Clean Data
Loại bỏ đơn hủy và gian lận, tạo cột `Profit_Margin` = `Order Item Profit Ratio` × 100.

In [ ]:
data_path = '/content/drive/MyDrive/DataCo SMART SUPPLY CHAIN FOR BIG DATA ANALYSIS/DataCoSupplyChainDataset.csv'
df = pd.read_csv(data_path, encoding='ISO-8859-1', low_memory=False)

df_clean = df[~df['Order Status'].isin(['CANCELED', 'SUSPECTED_FRAUD'])].copy()

# Cột phân tích chính
df_clean['Profit_Margin']   = df_clean['Order Item Profit Ratio'] * 100      # %
df_clean['Discount_Rate']   = df_clean['Order Item Discount Rate'] * 100     # %
df_clean['Profit_Per_Order']= df_clean['Order Profit Per Order']
df_clean['Sales']           = df_clean['Sales']

print(f'Shape sau cleaning : {df_clean.shape}')
print(f'Customer Segments  : {sorted(df_clean["Customer Segment"].unique())}')
print(f'Markets            : {sorted(df_clean["Market"].unique())}')
df_clean[['Profit_Margin', 'Discount_Rate', 'Profit_Per_Order', 'Sales']].describe().round(2)


## Bước 3: Phân tích Lợi nhuận theo Customer Segment

So sánh 3 phân khúc khách hàng: **Consumer**, **Corporate**, **Home Office** theo:
- Tổng lợi nhuận (Total Profit)
- Biên lợi nhuận trung bình (Avg Profit Margin %)
- Doanh thu trung bình / đơn hàng
- Mức chiết khấu trung bình đang áp dụng

In [ ]:
seg_stats = df_clean.groupby('Customer Segment').agg(
    Total_Profit    = ('Profit_Per_Order', 'sum'),
    Avg_Margin_Pct  = ('Profit_Margin',    'mean'),
    Avg_Sales       = ('Sales',            'mean'),
    Avg_Discount    = ('Discount_Rate',    'mean'),
    Order_Count     = ('Order Id',         'count')
).reset_index().sort_values('Total_Profit', ascending=False)

seg_stats['Avg_Profit_Per_Order'] = seg_stats['Total_Profit'] / seg_stats['Order_Count']
print('\n=== Thống kê theo Customer Segment ===')
display(seg_stats.round(2))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
palette_seg = ['#2196F3', '#FF9800', '#4CAF50']

# 1. Tổng lợi nhuận
bars = axes[0,0].bar(seg_stats['Customer Segment'], seg_stats['Total_Profit'],
                     color=palette_seg)
axes[0,0].bar_label(bars, fmt='${:,.0f}', padding=3, fontsize=10)
axes[0,0].set_title('Tổng Lợi nhuận theo Segment', fontweight='bold')
axes[0,0].set_ylabel('Tổng lợi nhuận ($)')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# 2. Biên lợi nhuận %
bars2 = axes[0,1].bar(seg_stats['Customer Segment'], seg_stats['Avg_Margin_Pct'],
                      color=palette_seg)
axes[0,1].bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=10)
axes[0,1].set_title('Biên Lợi nhuận TB (%) theo Segment', fontweight='bold')
axes[0,1].set_ylabel('Profit Margin (%)')
axes[0,1].axhline(y=df_clean['Profit_Margin'].mean(), color='red',
                  linestyle='--', alpha=0.7, label='TB toàn công ty')
axes[0,1].legend(fontsize=9)

# 3. Doanh thu TB / đơn
bars3 = axes[1,0].bar(seg_stats['Customer Segment'], seg_stats['Avg_Sales'],
                      color=palette_seg)
axes[1,0].bar_label(bars3, fmt='${:,.1f}', padding=3, fontsize=10)
axes[1,0].set_title('Doanh thu TB / Đơn hàng theo Segment', fontweight='bold')
axes[1,0].set_ylabel('Avg Sales ($)')

# 4. Chiết khấu TB
bars4 = axes[1,1].bar(seg_stats['Customer Segment'], seg_stats['Avg_Discount'],
                      color=palette_seg)
axes[1,1].bar_label(bars4, fmt='%.1f%%', padding=3, fontsize=10)
axes[1,1].set_title('Chiết khấu TB (%) theo Segment', fontweight='bold')
axes[1,1].set_ylabel('Avg Discount Rate (%)')

plt.suptitle('Phân tích Lợi nhuận theo Customer Segment', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## Bước 4: Phân tích Lợi nhuận theo Market

So sánh 5 thị trường: **Africa**, **Europe**, **LATAM**, **Pacific Asia**, **USCA**.

In [ ]:
mkt_stats = df_clean.groupby('Market').agg(
    Total_Profit    = ('Profit_Per_Order', 'sum'),
    Avg_Margin_Pct  = ('Profit_Margin',    'mean'),
    Avg_Sales       = ('Sales',            'mean'),
    Avg_Discount    = ('Discount_Rate',    'mean'),
    Order_Count     = ('Order Id',         'count')
).reset_index().sort_values('Total_Profit', ascending=False)

mkt_stats['Profit_Per_Order_Avg'] = mkt_stats['Total_Profit'] / mkt_stats['Order_Count']
print('\n=== Thống kê theo Market ===')
display(mkt_stats.round(2))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
palette_mkt = ['#E91E63', '#9C27B0', '#2196F3', '#FF9800', '#4CAF50']
markets_sorted = mkt_stats['Market'].tolist()

# 1. Tổng lợi nhuận
bars = axes[0,0].bar(mkt_stats['Market'], mkt_stats['Total_Profit'], color=palette_mkt)
axes[0,0].bar_label(bars, fmt='${:,.0f}', padding=3, fontsize=9)
axes[0,0].set_title('Tổng Lợi nhuận theo Market', fontweight='bold')
axes[0,0].set_ylabel('Tổng lợi nhuận ($)')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0,0].tick_params(axis='x', rotation=15)

# 2. Biên lợi nhuận %
bars2 = axes[0,1].bar(mkt_stats['Market'], mkt_stats['Avg_Margin_Pct'], color=palette_mkt)
axes[0,1].bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=9)
axes[0,1].set_title('Biên Lợi nhuận TB (%) theo Market', fontweight='bold')
axes[0,1].set_ylabel('Profit Margin (%)')
axes[0,1].axhline(y=df_clean['Profit_Margin'].mean(), color='red',
                  linestyle='--', alpha=0.7, label='TB toàn công ty')
axes[0,1].legend(fontsize=9)
axes[0,1].tick_params(axis='x', rotation=15)

# 3. Tỷ trọng số đơn hàng (Pie)
axes[1,0].pie(mkt_stats['Order_Count'], labels=mkt_stats['Market'],
              colors=palette_mkt, autopct='%1.1f%%', startangle=90,
              textprops={'fontsize': 10})
axes[1,0].set_title('Tỷ trọng Đơn hàng theo Market', fontweight='bold')

# 4. Lợi nhuận TB / đơn hàng
bars4 = axes[1,1].bar(mkt_stats['Market'], mkt_stats['Profit_Per_Order_Avg'], color=palette_mkt)
axes[1,1].bar_label(bars4, fmt='${:,.2f}', padding=3, fontsize=9)
axes[1,1].set_title('Lợi nhuận TB / Đơn hàng theo Market', fontweight='bold')
axes[1,1].set_ylabel('Avg Profit / Order ($)')
axes[1,1].axhline(y=df_clean['Profit_Per_Order'].mean(), color='red',
                  linestyle='--', alpha=0.7, label='TB toàn công ty')
axes[1,1].legend(fontsize=9)
axes[1,1].tick_params(axis='x', rotation=15)

plt.suptitle('Phân tích Lợi nhuận theo Market', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


### Heatmap: Lợi nhuận TB theo Segment × Market

In [ ]:
pivot_margin = df_clean.pivot_table(
    values='Profit_Margin',
    index='Customer Segment',
    columns='Market',
    aggfunc='mean'
).round(2)

pivot_profit = df_clean.pivot_table(
    values='Profit_Per_Order',
    index='Customer Segment',
    columns='Market',
    aggfunc='sum'
).round(0)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.heatmap(pivot_margin, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=axes[0], linewidths=0.5, cbar_kws={'label': 'Profit Margin (%)'})
axes[0].set_title('Biên LN TB (%) — Segment × Market', fontweight='bold')

sns.heatmap(pivot_profit, annot=True, fmt='.0f', cmap='Blues',
            ax=axes[1], linewidths=0.5, cbar_kws={'label': 'Total Profit ($)'})
axes[1].set_title('Tổng Lợi nhuận ($) — Segment × Market', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n=== Biên LN TB (%) theo Segment × Market ===')
display(pivot_margin)


## Bước 5: Phân tích Tác động Chiết khấu — Regression

Dùng **Polynomial Regression (bậc 2)** để mô hình hóa mối quan hệ phi tuyến giữa
`Discount Rate` và `Profit Margin`. Bậc 2 phù hợp vì:
- Chiết khấu thấp → margin ổn
- Chiết khấu tăng cao → margin giảm mạnh (quan hệ lõm, hình chữ U ngược)

Phân tích riêng cho từng **Customer Segment** để tìm điểm chiết khấu tối ưu.

In [ ]:
# Scatter + Regression tổng thể
fig, ax = plt.subplots(figsize=(12, 6))

# Sample để vẽ nhanh
sample = df_clean[['Discount_Rate', 'Profit_Margin']].dropna().sample(
    min(5000, len(df_clean)), random_state=42)

ax.scatter(sample['Discount_Rate'], sample['Profit_Margin'],
           alpha=0.3, s=15, color='steelblue', label='Giao dịch thực tế')

# Polynomial Regression bậc 2
X_all = df_clean[['Discount_Rate']].dropna()
y_all = df_clean.loc[X_all.index, 'Profit_Margin']
model_poly = make_pipeline(PolynomialFeatures(2), LinearRegression())
model_poly.fit(X_all, y_all)

x_line = np.linspace(X_all['Discount_Rate'].min(), X_all['Discount_Rate'].max(), 200).reshape(-1,1)
x_line_df = pd.DataFrame(x_line, columns=['Discount_Rate'])
y_line = model_poly.predict(x_line_df)

ax.plot(x_line, y_line, color='red', linewidth=2.5, label='Polynomial Regression (bậc 2)')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='Breakeven (Margin=0)')

r2 = r2_score(y_all, model_poly.predict(X_all))
ax.set_title(f'Tác động Discount Rate lên Profit Margin (R²={r2:.3f})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Discount Rate (%)')
ax.set_ylabel('Profit Margin (%)')
ax.legend()
plt.tight_layout()
plt.show()
print(f'R² toàn bộ: {r2:.4f}')


In [ ]:
# Regression riêng cho từng Segment
segments = df_clean['Customer Segment'].unique()
palette_seg = {'Consumer': '#2196F3', 'Corporate': '#FF9800', 'Home Office': '#4CAF50'}

seg_models = {}     # lưu model để dùng trong LP
seg_optimal = {}    # mức discount tối ưu từ regression

fig, axes = plt.subplots(1, len(segments), figsize=(17, 5), sharey=True)

for ax, seg in zip(axes, segments):
    seg_df = df_clean[df_clean['Customer Segment'] == seg][['Discount_Rate', 'Profit_Margin']].dropna()
    X_seg  = seg_df[['Discount_Rate']]
    y_seg  = seg_df['Profit_Margin']

    m = make_pipeline(PolynomialFeatures(2), LinearRegression())
    m.fit(X_seg, y_seg)
    seg_models[seg] = m

    x_line = np.linspace(0, seg_df['Discount_Rate'].max(), 300).reshape(-1,1)
    x_df   = pd.DataFrame(x_line, columns=['Discount_Rate'])
    y_line = m.predict(x_df)

    # Tìm đỉnh (optimal discount) từ regression
    opt_idx  = np.argmax(y_line)
    opt_disc = float(x_line[opt_idx])
    opt_marg = float(y_line[opt_idx])
    seg_optimal[seg] = {'opt_discount': round(opt_disc, 2), 'pred_margin': round(opt_marg, 2)}

    sample_s = seg_df.sample(min(2000, len(seg_df)), random_state=42)
    ax.scatter(sample_s['Discount_Rate'], sample_s['Profit_Margin'],
               alpha=0.3, s=12, color=palette_seg.get(seg, 'gray'))
    ax.plot(x_line, y_line, color='red', linewidth=2.5)
    ax.axvline(x=opt_disc, color='darkgreen', linestyle='--', linewidth=2,
               label=f'Opt. Discount={opt_disc:.1f}%')
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'{seg}\n(opt={opt_disc:.1f}%, pred margin={opt_marg:.1f}%)',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Discount Rate (%)')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Profit Margin (%)')
plt.suptitle('Regression: Discount Rate vs Profit Margin theo Segment',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Discount Rate tối ưu từ Regression (Grid Search trên curve) ===')
for seg, v in seg_optimal.items():
    print(f'  {seg:15s}: Discount tối ưu = {v["opt_discount"]:5.1f}%  |  Pred. Margin = {v["pred_margin"]:6.2f}%')


## Bước 6: Tối ưu hóa Discount Rate — Linear Programming

**Bài toán LP**: Tìm mức discount cho từng Segment để **tối đa hóa tổng lợi nhuận ròng**,
với các ràng buộc:

| Ràng buộc | Lý do |
|-----------|-------|
| 0% ≤ Discount ≤ 30% | Không chiết khấu âm; giới hạn trên 30% vì qua đó margin âm |
| Margin ≥ 5% (mỗi Segment) | Đảm bảo không bán lỗ |
| Weighted avg discount ≤ 15% | Chính sách tài chính tổng thể |

> **Linearization**: Vì Profit Margin là hàm phi tuyến của Discount Rate,
> ta xấp xỉ tuyến tính quanh điểm tối ưu từ Regression (linearization workaround).
> Slope tuyến tính = đạo hàm của polynomial regression tại điểm tham chiếu.


In [ ]:
# ── Tính toán hệ số LP từ Regression ─────────────────────────────────────────

# Tham chiếu: discount hiện tại của từng segment
disc_current = df_clean.groupby('Customer Segment')['Discount_Rate'].mean().to_dict()
orders_count = df_clean.groupby('Customer Segment')['Order Id'].count().to_dict()
total_orders = sum(orders_count.values())
weights      = {s: orders_count[s] / total_orders for s in segments}

# Với mỗi segment: tính margin tại discount hiện tại và slope quanh đó
lp_data = {}
delta   = 1.0   # bước tính đạo hàm số (1%)

for seg in segments:
    d0    = disc_current[seg]
    model = seg_models[seg]
    m0    = float(model.predict(pd.DataFrame([[d0]], columns=['Discount_Rate']))[0])
    m_p   = float(model.predict(pd.DataFrame([[d0+delta]], columns=['Discount_Rate']))[0])
    slope = (m_p - m0) / delta   # dMargin/dDiscount tại d0

    lp_data[seg] = {
        'd0': d0, 'm0': m0, 'slope': slope, 'weight': weights[seg],
        'avg_sales': df_clean[df_clean['Customer Segment']==seg]['Sales'].mean()
    }
    print(f'{seg:15s}: d0={d0:.1f}%, margin@d0={m0:.2f}%, slope={slope:.4f}%/% discount')


In [ ]:
# ── Thiết lập bài toán LP ─────────────────────────────────────────────────────
# Biến: x = [Δd_Consumer, Δd_Corporate, Δd_HomeOffice]
#        tức là thay đổi discount so với d0 hiện tại
# Mục tiêu: MAXIMIZE Σ weight_i × (m0_i + slope_i × Δd_i)
#          = MINIMIZE Σ -weight_i × slope_i × Δd_i  (vì linprog minimize)

seg_list = list(segments)
n = len(seg_list)

# Objective: -weight * slope (để maximize margin)
c = np.array([-lp_data[s]['weight'] * lp_data[s]['slope'] for s in seg_list])

# Ràng buộc bất đẳng thức (A_ub @ x <= b_ub)
A_ub = []
b_ub = []

# 1. Margin tối thiểu 5% cho từng segment:
#    m0_i + slope_i * Δd_i >= 5  ⟺  -slope_i * Δd_i <= m0_i - 5
for seg in seg_list:
    row = [0.0] * n
    idx = seg_list.index(seg)
    row[idx] = -lp_data[seg]['slope']
    A_ub.append(row)
    b_ub.append(lp_data[seg]['m0'] - 5.0)

# 2. Weighted avg discount <= 15%:
#    Σ weight_i * (d0_i + Δd_i) <= 15
#    Σ weight_i * Δd_i <= 15 - Σ weight_i * d0_i
row_avg = [lp_data[s]['weight'] for s in seg_list]
b_avg   = 15.0 - sum(lp_data[s]['weight'] * lp_data[s]['d0'] for s in seg_list)
A_ub.append(row_avg)
b_ub.append(b_avg)

A_ub = np.array(A_ub)
b_ub = np.array(b_ub)

# Bounds: Discount tổng phải trong [0, 30]
#  d0_i + Δd_i ∈ [0, 30]  →  Δd_i ∈ [-d0_i, 30 - d0_i]
bounds = [(-lp_data[s]['d0'], 30.0 - lp_data[s]['d0']) for s in seg_list]

# Giải LP
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

print(f'LP Status: {result.message}')
print(f'Optimal objective (max weighted margin): {-result.fun:.4f}%')
print()

lp_results = []
for i, seg in enumerate(seg_list):
    delta_d      = result.x[i]
    opt_disc_lp  = lp_data[seg]['d0'] + delta_d
    pred_margin  = lp_data[seg]['m0'] + lp_data[seg]['slope'] * delta_d
    lp_results.append({
        'Segment': seg,
        'Discount Hiện tại (%)': round(lp_data[seg]['d0'], 2),
        'Discount Tối ưu LP (%)': round(opt_disc_lp, 2),
        'Thay đổi (pp)': round(delta_d, 2),
        'Margin Hiện tại (%)': round(lp_data[seg]['m0'], 2),
        'Pred. Margin LP (%)': round(pred_margin, 2)
    })
    print(f'{seg:15s}: d_opt={opt_disc_lp:.2f}%  (Δ={delta_d:+.2f}pp) | Pred. Margin={pred_margin:.2f}%')

lp_df = pd.DataFrame(lp_results)
display(lp_df)


In [ ]:
# Biểu đồ so sánh Discount hiện tại vs Tối ưu LP
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
x = np.arange(len(lp_results))
w = 0.35

curr_disc = [r['Discount Hiện tại (%)'] for r in lp_results]
opt_disc  = [r['Discount Tối ưu LP (%)'] for r in lp_results]
curr_marg = [r['Margin Hiện tại (%)'] for r in lp_results]
pred_marg = [r['Pred. Margin LP (%)'] for r in lp_results]
seg_names = [r['Segment'] for r in lp_results]

# Discount comparison
b1 = axes[0].bar(x - w/2, curr_disc, w, label='Discount Hiện tại', color='steelblue', alpha=0.8)
b2 = axes[0].bar(x + w/2, opt_disc,  w, label='Discount Tối ưu LP', color='darkorange', alpha=0.8)
axes[0].bar_label(b1, fmt='%.1f%%', padding=3, fontsize=10)
axes[0].bar_label(b2, fmt='%.1f%%', padding=3, fontsize=10)
axes[0].set_xticks(x)
axes[0].set_xticklabels(seg_names, rotation=10)
axes[0].set_ylabel('Discount Rate (%)')
axes[0].set_title('Discount Rate: Hiện tại vs Tối ưu LP', fontweight='bold')
axes[0].legend()

# Margin comparison
b3 = axes[1].bar(x - w/2, curr_marg, w, label='Margin Hiện tại', color='steelblue', alpha=0.8)
b4 = axes[1].bar(x + w/2, pred_marg, w, label='Pred. Margin LP', color='seagreen', alpha=0.8)
axes[1].bar_label(b3, fmt='%.2f%%', padding=3, fontsize=10)
axes[1].bar_label(b4, fmt='%.2f%%', padding=3, fontsize=10)
axes[1].set_xticks(x)
axes[1].set_xticklabels(seg_names, rotation=10)
axes[1].set_ylabel('Profit Margin (%)')
axes[1].set_title('Profit Margin: Hiện tại vs Dự báo sau LP', fontweight='bold')
axes[1].legend()

plt.suptitle('Kết quả Tối ưu hóa Linear Programming — Discount Rate',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Bước 7: Sensitivity Analysis — Phân tích Độ nhạy

Mô phỏng tác động của các mức Discount khác nhau lên Profit Margin cho từng Segment,
để kiểm tra tính ổn định của giải pháp LP.

In [ ]:
fig, axes = plt.subplots(1, len(seg_list), figsize=(17, 5), sharey=True)
disc_grid = np.linspace(0, 30, 200).reshape(-1,1)
disc_grid_df = pd.DataFrame(disc_grid, columns=['Discount_Rate'])

for ax, seg in zip(axes, seg_list):
    model   = seg_models[seg]
    margins = model.predict(disc_grid_df)

    ax.plot(disc_grid, margins, color='steelblue', linewidth=2.5)
    ax.fill_between(disc_grid.flatten(), margins, alpha=0.1, color='steelblue')

    # Đường ràng buộc margin >= 5%
    ax.axhline(y=5, color='red', linestyle='--', linewidth=1.5, label='Min Margin 5%')
    ax.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.7)

    # Điểm LP optimal
    opt = next(r for r in lp_results if r['Segment'] == seg)
    ax.axvline(x=opt['Discount Tối ưu LP (%)'], color='darkorange', linestyle='--',
               linewidth=2, label=f"LP opt={opt['Discount Tối ưu LP (%)']:.1f}%")

    # Điểm hiện tại
    ax.axvline(x=lp_data[seg]['d0'], color='purple', linestyle=':',
               linewidth=2, label=f"Current={lp_data[seg]['d0']:.1f}%")

    ax.set_title(f'{seg}', fontweight='bold')
    ax.set_xlabel('Discount Rate (%)')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 30)

axes[0].set_ylabel('Profit Margin (%)')
plt.suptitle('Sensitivity Analysis: Discount Rate vs Profit Margin theo Segment',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Bước 9: Executive Summary & Khuyến nghị Hành động

## Bước 8 (nâng cao): Mô hình Cân bằng Discount–Revenue–Profit

### Tư duy bài toán

Bài toán gốc: **Tối đa hóa Lợi nhuận Ròng** — không chỉ Profit Margin (%).

```
Net Profit = Sales Volume × Unit Price × Profit Margin
           = f(Discount) × g(Discount)
```

| Chiều tác động | Khi Discount tăng | Hệ quả |
|----------------|-------------------|--------|
| Sales Volume   | ↑ Tăng (khách mua nhiều hơn) | Revenue tăng |
| Profit Margin  | ↓ Giảm (biên lãi mỏng đi) | Margin giảm |
| **Net Profit** | **Có điểm tối ưu dạng hình chữ U ngược** | ← Cần tìm đỉnh này |

**Phương pháp:**
1. **Regression** `Discount → Volume` và `Discount → Margin` (đã có)
2. **Grid Search** (0→30%): tính Net Profit tại từng mức → vẽ biểu đồ trực quan
3. **Non-linear Optimization** (`scipy.optimize`): tìm điểm cực đại chính xác


### 8.1 — Regression: Discount Rate → Sales Volume

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize_scalar

# Gộp theo Order Id: mỗi đơn có 1 discount rate và 1 quantity
order_level = df_clean.groupby('Order Id').agg(
    Discount_Rate       = ('Discount_Rate',         'mean'),
    Order_Item_Quantity = ('Order Item Quantity',    'sum'),
    Sales               = ('Sales',                 'sum'),
    Profit_Margin       = ('Profit_Margin',         'mean'),
    Profit_Per_Order    = ('Profit_Per_Order',       'mean'),
    Avg_Unit_Price      = ('Order Item Product Price','mean'),
).reset_index()

order_level = order_level[order_level['Discount_Rate'] >= 0].dropna()
print(f'Số đơn hàng phân tích: {len(order_level):,}')
order_level[['Discount_Rate','Order_Item_Quantity','Sales','Profit_Margin']].describe().round(2)


In [ ]:
# Regression bậc 2: Discount → Sales Volume (số lượng/đơn)
X_vol = order_level[['Discount_Rate']]
y_vol = order_level['Order_Item_Quantity']
model_volume = make_pipeline(PolynomialFeatures(2), LinearRegression())
model_volume.fit(X_vol, y_vol)

# Regression bậc 2: Discount → Profit Margin
X_mar = order_level[['Discount_Rate']]
y_mar = order_level['Profit_Margin']
model_margin_order = make_pipeline(PolynomialFeatures(2), LinearRegression())
model_margin_order.fit(X_mar, y_mar)

# Avg Unit Price (giả định không đổi nhiều theo discount trong ngắn hạn)
avg_unit_price = order_level['Avg_Unit_Price'].mean()

# Kiểm tra R²
from sklearn.metrics import r2_score
r2_vol = r2_score(y_vol, model_volume.predict(X_vol))
r2_mar = r2_score(y_mar, model_margin_order.predict(X_mar))
print(f'R² (Volume  ~ Discount): {r2_vol:.4f}')
print(f'R² (Margin  ~ Discount): {r2_mar:.4f}')
print(f'Avg Unit Price: ${avg_unit_price:.2f}')


In [ ]:
# Vẽ 2 regression cùng nhau để hiểu chiều tác động
disc_vals = np.linspace(0, 30, 300).reshape(-1,1)
disc_df   = pd.DataFrame(disc_vals, columns=['Discount_Rate'])

pred_vol = model_volume.predict(disc_df)
pred_mar = model_margin_order.predict(disc_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Volume curve
sample_v = order_level.sample(min(3000, len(order_level)), random_state=42)
axes[0].scatter(sample_v['Discount_Rate'], sample_v['Order_Item_Quantity'],
                alpha=0.25, s=15, color='steelblue')
axes[0].plot(disc_vals, pred_vol, color='red', linewidth=2.5,
             label=f'Poly Regression (R²={r2_vol:.3f})')
axes[0].set_title('Discount Rate → Sales Volume (qty/order)', fontweight='bold')
axes[0].set_xlabel('Discount Rate (%)')
axes[0].set_ylabel('Số lượng / đơn hàng')
axes[0].legend()

# Margin curve
axes[1].scatter(sample_v['Discount_Rate'], sample_v['Profit_Margin'],
                alpha=0.25, s=15, color='darkorange')
axes[1].plot(disc_vals, pred_mar, color='red', linewidth=2.5,
             label=f'Poly Regression (R²={r2_mar:.3f})')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Discount Rate → Profit Margin (%)', fontweight='bold')
axes[1].set_xlabel('Discount Rate (%)')
axes[1].set_ylabel('Profit Margin (%)')
axes[1].legend()

plt.suptitle('Hai chiều tác động của Discount Rate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 8.2 — Grid Search: Tính Net Profit tại từng mức Discount (0→30%)

```
Net Profit per Order = Volume(d) × Unit_Price × Margin(d) / 100
```


In [ ]:
# Grid Search: thử 301 mức discount từ 0% đến 30%
disc_grid  = np.linspace(0, 30, 301)
disc_grid_df = pd.DataFrame(disc_grid, columns=['Discount_Rate'])

pred_volumes = np.maximum(model_volume.predict(disc_grid_df), 0.01)   # volume >= 0
pred_margins = model_margin_order.predict(disc_grid_df)                # margin có thể âm

# Net Profit = Volume × UnitPrice × (Margin/100)
net_profit_curve = pred_volumes * avg_unit_price * (pred_margins / 100)

# Tìm đỉnh Grid Search
best_idx_gs      = np.argmax(net_profit_curve)
best_disc_gs     = disc_grid[best_idx_gs]
best_vol_gs      = pred_volumes[best_idx_gs]
best_margin_gs   = pred_margins[best_idx_gs]
best_profit_gs   = net_profit_curve[best_idx_gs]

print(f'=== Grid Search Result ===')
print(f'Discount Rate tối ưu : {best_disc_gs:.2f}%')
print(f'Pred. Volume         : {best_vol_gs:.2f} đơn vị/đơn')
print(f'Pred. Margin         : {best_margin_gs:.2f}%')
print(f'Net Profit / Order   : ${best_profit_gs:.4f}')


In [ ]:
# Vẽ biểu đồ Net Profit curve
fig, axes = plt.subplots(3, 1, figsize=(14, 14))

# 1. Volume curve
axes[0].plot(disc_grid, pred_volumes, color='steelblue', linewidth=2.5)
axes[0].axvline(x=best_disc_gs, color='red', linestyle='--', linewidth=2,
                label=f'Optimal = {best_disc_gs:.1f}%')
axes[0].fill_between(disc_grid, pred_volumes, alpha=0.1, color='steelblue')
axes[0].set_title('Predicted Sales Volume theo Discount Rate', fontweight='bold')
axes[0].set_ylabel('Số lượng / đơn hàng')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.2f}'))

# 2. Margin curve
axes[1].plot(disc_grid, pred_margins, color='darkorange', linewidth=2.5)
axes[1].axvline(x=best_disc_gs, color='red', linestyle='--', linewidth=2,
                label=f'Optimal = {best_disc_gs:.1f}%')
axes[1].axhline(y=0, color='gray', linestyle=':', alpha=0.7, label='Breakeven')
axes[1].fill_between(disc_grid, pred_margins,
                     where=(np.array(pred_margins) >= 0),
                     alpha=0.15, color='green', label='Margin dương')
axes[1].fill_between(disc_grid, pred_margins,
                     where=(np.array(pred_margins) < 0),
                     alpha=0.15, color='red', label='Margin âm')
axes[1].set_title('Predicted Profit Margin (%) theo Discount Rate', fontweight='bold')
axes[1].set_ylabel('Profit Margin (%)')
axes[1].legend()

# 3. Net Profit curve (đỉnh = optimal)
positive_mask = net_profit_curve >= 0
axes[2].plot(disc_grid, net_profit_curve, color='seagreen', linewidth=3)
axes[2].fill_between(disc_grid, net_profit_curve,
                     where=positive_mask, alpha=0.2, color='seagreen', label='Net Profit dương')
axes[2].fill_between(disc_grid, net_profit_curve,
                     where=~positive_mask, alpha=0.2, color='red', label='Net Profit âm')
axes[2].axvline(x=best_disc_gs, color='red', linestyle='--', linewidth=2.5,
                label=f'⭐ Optimal Discount = {best_disc_gs:.1f}%')
axes[2].axhline(y=0, color='gray', linestyle=':', alpha=0.7)
axes[2].scatter([best_disc_gs], [best_profit_gs],
                color='red', s=150, zorder=5, label=f'Max Net Profit = ${best_profit_gs:.4f}')
axes[2].set_title('NET PROFIT / Order theo Discount Rate — Tìm Điểm Tối ưu',
                  fontweight='bold', fontsize=13)
axes[2].set_xlabel('Discount Rate (%)')
axes[2].set_ylabel('Net Profit / Order ($)')
axes[2].legend()

for ax in axes:
    ax.set_xlim(0, 30)
    ax.tick_params(axis='x', which='major')

plt.suptitle('Grid Search: Tìm Điểm Cân bằng Discount–Revenue–Profit',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 8.3 — Non-linear Optimization: Tìm Điểm Tối ưu Chính xác

Dùng `scipy.optimize.minimize_scalar` (Bounded Brent Method) để tìm
chính xác mức Discount tối đa hóa Net Profit trong khoảng [0%, 30%].

In [ ]:
# Hàm Net Profit (âm vì minimize_scalar minimize)
def neg_net_profit(discount):
    d_df   = pd.DataFrame([[discount]], columns=['Discount_Rate'])
    vol    = max(model_volume.predict(d_df)[0], 0.01)
    margin = model_margin_order.predict(d_df)[0]
    return -(vol * avg_unit_price * margin / 100)

# Tối ưu hóa: Bounded Brent Method trong [0, 30]
opt_result = minimize_scalar(neg_net_profit, bounds=(0, 30), method='bounded')

opt_discount = opt_result.x
opt_d_df     = pd.DataFrame([[opt_discount]], columns=['Discount_Rate'])
opt_volume   = max(model_volume.predict(opt_d_df)[0], 0)
opt_margin   = model_margin_order.predict(opt_d_df)[0]
opt_profit   = opt_volume * avg_unit_price * opt_margin / 100

print(f'=== Non-linear Optimization (Bounded Brent) ===')
print(f'Convergence     : {opt_result.message}')
print(f'Discount tối ưu : {opt_discount:.4f}%')
print(f'Pred. Volume    : {opt_volume:.4f} đơn vị/đơn')
print(f'Pred. Margin    : {opt_margin:.4f}%')
print(f'Max Net Profit  : ${opt_profit:.6f}/order')
print()
print(f'--- Validation: Grid Search vs Non-linear ---')
print(f'Grid Search opt : {best_disc_gs:.2f}%')
print(f'NL Optimizer opt: {opt_discount:.4f}%')
print(f'Chênh lệch      : {abs(opt_discount - best_disc_gs):.4f}pp '
      f'({"✅ Hợp lệ" if abs(opt_discount - best_disc_gs) < 1 else "⚠️ Cần xem lại"})')


### 8.4 — Phân tích Net Profit Optimization theo từng Customer Segment

Chạy lại toàn bộ pipeline Grid Search + Non-linear Optimization **riêng cho từng Segment**
để tìm mức discount tối ưu cá nhân hóa hơn.

In [ ]:
seg_optimization_results = []
palette_seg = {'Consumer': '#2196F3', 'Corporate': '#FF9800', 'Home Office': '#4CAF50'}

fig, axes = plt.subplots(len(seg_list), 1, figsize=(14, 5 * len(seg_list)))
if len(seg_list) == 1:
    axes = [axes]

for ax, seg in zip(axes, seg_list):
    seg_orders = order_level[order_level['Order Id'].isin(
        df_clean[df_clean['Customer Segment'] == seg]['Order Id']
    )] if 'Order Id' in order_level.columns else order_level   # fallback

    # Nếu filter đúng, dùng data riêng của segment
    seg_orders = df_clean[df_clean['Customer Segment'] == seg].groupby('Order Id').agg(
        Discount_Rate       = ('Discount_Rate',          'mean'),
        Order_Item_Quantity = ('Order Item Quantity',     'sum'),
        Profit_Margin       = ('Profit_Margin',          'mean'),
        Avg_Unit_Price      = ('Order Item Product Price','mean'),
    ).reset_index().dropna()

    avg_price_seg = seg_orders['Avg_Unit_Price'].mean()

    # Train models riêng
    X_v = seg_orders[['Discount_Rate']]; y_v = seg_orders['Order_Item_Quantity']
    X_m = seg_orders[['Discount_Rate']]; y_m = seg_orders['Profit_Margin']
    m_vol = make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(X_v, y_v)
    m_mar = make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(X_m, y_m)

    # Grid Search
    d_grid   = np.linspace(0, 30, 301)
    d_gdf    = pd.DataFrame(d_grid, columns=['Discount_Rate'])
    vols_g   = np.maximum(m_vol.predict(d_gdf), 0.01)
    mars_g   = m_mar.predict(d_gdf)
    nps_g    = vols_g * avg_price_seg * (mars_g / 100)

    best_idx = np.argmax(nps_g)
    gs_disc  = d_grid[best_idx]
    gs_np    = nps_g[best_idx]

    # Non-linear Opt
    def neg_np_seg(d):
        d_df = pd.DataFrame([[d]], columns=['Discount_Rate'])
        v = max(m_vol.predict(d_df)[0], 0.01)
        m = m_mar.predict(d_df)[0]
        return -(v * avg_price_seg * m / 100)

    res_nl  = minimize_scalar(neg_np_seg, bounds=(0, 30), method='bounded')
    nl_disc = res_nl.x
    nl_np   = -res_nl.fun

    # Current discount avg
    curr_d = seg_orders['Discount_Rate'].mean()
    curr_d_df = pd.DataFrame([[curr_d]], columns=['Discount_Rate'])
    curr_np = (max(m_vol.predict(curr_d_df)[0], 0.01)
               * avg_price_seg
               * m_mar.predict(curr_d_df)[0] / 100)

    seg_optimization_results.append({
        'Segment': seg,
        'Discount Hiện tại (%)': round(curr_d, 2),
        'Discount Tối ưu GS (%)': round(gs_disc, 2),
        'Discount Tối ưu NL (%)': round(nl_disc, 4),
        'Net Profit Hiện tại ($)': round(curr_np, 4),
        'Max Net Profit NL ($)': round(nl_np, 4),
        'Cải thiện NP (%)': round((nl_np - curr_np) / abs(curr_np) * 100, 2) if curr_np != 0 else 0
    })

    color = palette_seg.get(seg, 'steelblue')
    ax.plot(d_grid, nps_g, color=color, linewidth=3, label='Net Profit Curve')
    ax.fill_between(d_grid, nps_g, where=(nps_g >= 0), alpha=0.15, color=color)
    ax.fill_between(d_grid, nps_g, where=(nps_g < 0),  alpha=0.15, color='red')
    ax.axvline(x=nl_disc, color='red', linestyle='--', linewidth=2.5,
               label=f'⭐ NL Optimal = {nl_disc:.1f}%')
    ax.axvline(x=curr_d,  color='gray', linestyle=':',  linewidth=2,
               label=f'Current avg = {curr_d:.1f}%')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
    ax.scatter([nl_disc], [nl_np], color='red', s=150, zorder=5)
    ax.set_title(f'{seg} — Net Profit / Order vs Discount Rate', fontweight='bold')
    ax.set_ylabel('Net Profit / Order ($)')
    ax.set_xlim(0, 30)
    ax.legend(fontsize=9)

axes[-1].set_xlabel('Discount Rate (%)')
plt.suptitle('Tối ưu hóa Net Profit theo từng Customer Segment\n(Grid Search + Non-linear Optimization)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Bảng tổng hợp kết quả tối ưu hóa
opt_summary_df = pd.DataFrame(seg_optimization_results)
display(opt_summary_df)

print('\n=== Kết quả Tối ưu hóa Discount-Revenue-Profit ===')
for r in seg_optimization_results:
    delta_d = r['Discount Tối ưu NL (%)'] - r['Discount Hiện tại (%)']
    action  = '↓ Giảm' if delta_d < -0.1 else ('↑ Tăng' if delta_d > 0.1 else '→ Giữ nguyên')
    arrow   = '📈' if r['Cải thiện NP (%)'] > 0 else '📉'
    print(f'\n  {r["Segment"]:15s}:')
    print(f'    Discount: {r["Discount Hiện tại (%)"]:5.1f}% → {r["Discount Tối ưu NL (%)"]:5.2f}%  ({action} {abs(delta_d):.1f}pp)')
    print(f'    Net Profit: ${r["Net Profit Hiện tại ($)"]:.4f} → ${r["Max Net Profit NL ($)"]:.4f}  '
          f'{arrow} ({r["Cải thiện NP (%)"]:+.1f}%)')


### 8.5 — So sánh 3 phương pháp Tối ưu hóa

| Phương pháp | Mục tiêu | Ưu điểm | Hạn chế |
|-------------|----------|---------|---------|
| **LP (Bước 6)** | Max Profit Margin % | Có ràng buộc chính sách, giải nhanh | Không tính volume effect |
| **Grid Search (8.2)** | Max Net Profit = Volume × Margin | Trực quan, dễ giải thích | Độ chính xác theo bước lưới |
| **NL Optimization (8.3)** | Max Net Profit = Volume × Margin | Chính xác nhất | Phụ thuộc chất lượng Regression |

> **Khuyến nghị**: Dùng **NL Optimization** làm giải pháp chính, **Grid Search** để trình bày trực quan cho ban lãnh đạo, **LP** để kiểm soát ràng buộc chính sách tổng thể.

In [ ]:
# Bảng so sánh 3 phương pháp cho toàn công ty
comparison_data = {
    'Phương pháp': ['LP Optimization (Bước 6)', 'Grid Search (Bước 8.2)', 'NL Optimization (Bước 8.3)'],
    'Mục tiêu': ['Max Profit Margin %', 'Max Net Profit/Order', 'Max Net Profit/Order'],
    'Discount tối ưu (%)': [
        f"{np.mean([r['Discount Tối ưu LP (%)'] for r in lp_results]):.1f}% (avg)",
        f'{best_disc_gs:.2f}%',
        f'{opt_discount:.2f}%'
    ],
    'Có tính Volume Effect': ['❌ Không', '✅ Có', '✅ Có'],
    'Có ràng buộc chính sách': ['✅ Có', '❌ Không', '❌ Không'],
}
comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)


In [ ]:
print()
print('=' * 70)
print('EXECUTIVE SUMMARY — PROFIT & DISCOUNT OPTIMIZATION (PHASE 3)')
print('=' * 70)

# Top Segment
best_seg = seg_stats.iloc[0]
print(f'\n🏆 Phân khúc sinh lời nhất: {best_seg["Customer Segment"]}')
print(f'   Tổng lợi nhuận  : ${best_seg["Total_Profit"]:>12,.0f}')
print(f'   Biên LN trung bình: {best_seg["Avg_Margin_Pct"]:>8.2f}%')

# Top Market
best_mkt = mkt_stats.iloc[0]
print(f'\n🌍 Thị trường sinh lời nhất: {best_mkt["Market"]}')
print(f'   Tổng lợi nhuận  : ${best_mkt["Total_Profit"]:>12,.0f}')
print(f'   Biên LN trung bình: {best_mkt["Avg_Margin_Pct"]:>8.2f}%')

# LP Results
print(f'\n📊 Kết quả Tối ưu hóa LP:')
for r in lp_results:
    delta  = r['Discount Tối ưu LP (%)'] - r['Discount Hiện tại (%)']
    action = '↓ Giảm' if delta < 0 else '↑ Tăng' if delta > 0 else '→ Giữ nguyên'
    margin_delta = r['Pred. Margin LP (%)'] - r['Margin Hiện tại (%)']
    print(f'   {r["Segment"]:15s}: {action} discount {abs(delta):.1f}pp '
          f'({r["Discount Hiện tại (%)"]:.1f}% → {r["Discount Tối ưu LP (%)"]:.1f}%) '
          f'| Margin: {r["Margin Hiện tại (%)"]:+.2f}% → {r["Pred. Margin LP (%)"]:+.2f}% '
          f'(Δ={margin_delta:+.2f}pp)')


## Khuyến nghị Hành động

### Về Phân khúc & Thị trường
1. **Tập trung nguồn lực vào phân khúc & thị trường sinh lời cao nhất** — dồn ngân sách marketing,
   dịch vụ hậu mãi và ưu tiên xử lý đơn hàng cho các segment/market có biên LN tốt nhất.
2. **Điều tra phân khúc có biên LN thấp**: Kiểm tra xem nguyên nhân là chi phí vận chuyển cao,
   giá sản phẩm thấp, hay discount quá cao — để có hành động can thiệp cụ thể.

### Về Chiết khấu
3. **Áp dụng mức Discount tối ưu từ LP**: Điều chỉnh chính sách chiết khấu theo từng Segment
   về mức LP optimal — đặc biệt tránh các chương trình khuyến mại vượt ngưỡng gây margin âm.
4. **Giới hạn trần discount tổng thể ≤ 15%** (ràng buộc LP) để đảm bảo lợi nhuận ròng
   toàn danh mục luôn dương.
5. **Discount theo tier khách hàng**: Corporate và Home Office thường mua số lượng lớn hơn —
   có thể áp dụng discount cao hơn nếu bù đắp bằng volume, nhưng cần kiểm soát margin tối thiểu.

---

## 🔍 Góc nhìn Phản biện

- **Linearization assumption**: LP sử dụng xấp xỉ tuyến tính quanh điểm d0 — kết quả chính xác nhất
  khi điều chỉnh discount không quá lớn (±5pp). Nếu thay đổi lớn hơn, cần dùng non-linear optimization.
- **Polynomial Regression bậc 2**: R² có thể thấp vì nhiều yếu tố khác ảnh hưởng đến margin
  (sản phẩm, khu vực, mùa vụ). Mô hình cần được bổ sung thêm biến để tăng giải thích lực.
- **Ràng buộc margin ≥ 5%**: Đây là tham số tùy chỉnh — nên xác nhận với CFO ngưỡng tối thiểu
  phù hợp với chiến lược tài chính công ty.
- **Kết hợp với Phase 1 & 2**: Phân khúc có discount cao thường cũng có Late Delivery Rate cao hơn
  (Phase 1). Giảm discount → giảm áp lực đơn hàng → có thể cải thiện đồng thời cả LDR và margin.
